In [ ]:
%run ../src/features/preprocess.py

In [ ]:
%run ../src/training/train.py

In [2]:
%run ../src/serving/export_model.py


Finding best MLP model in MLflow...
Best MLP run ID: afbc99eff4fc4c66af8e1b992cea4149
AUC-ROC: 0.9262
Exporting model to S3...


Uploaded: models/mlp_model/conda.yaml
Uploaded: models/mlp_model/MLmodel
Uploaded: models/mlp_model/python_env.yaml
Uploaded: models/mlp_model/requirements.txt
Uploaded: models/mlp_model/data/keras_module.txt
Uploaded: models/mlp_model/data/model.keras
Uploaded: models/mlp_model/data/save_format.txt
Model exported to S3 successfully

Model available at s3://fraud-detection-mlops-rd/models/mlp_model/


In [3]:
import tensorflow as tf
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = mlflow.tracking.MlflowClient()

# Load the model
experiment = client.get_experiment_by_name("fraud-detection")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'mlp_neural_network'",
    order_by=["metrics.auc_roc DESC"],
    max_results=1
)

run_id = runs[0].info.run_id
model = mlflow.tensorflow.load_model(f"runs:/{run_id}/mlp_model")

# Save in H5 format — compatible across Keras versions
model.save("models/mlp_model.h5")
print("Model saved in H5 format")

Model saved in H5 format


In [4]:
import boto3

s3_client = boto3.client("s3", region_name="us-east-2")
s3_client.upload_file(
    "models/mlp_model.h5",
    "fraud-detection-mlops-rd",
    "models/mlp_model.h5"
)
print("H5 model uploaded to S3")


H5 model uploaded to S3


In [5]:
import tensorflow as tf
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = mlflow.tracking.MlflowClient()

experiment = client.get_experiment_by_name("fraud-detection")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'mlp_neural_network'",
    order_by=["metrics.auc_roc DESC"],
    max_results=1
)

run_id = runs[0].info.run_id
model = mlflow.tensorflow.load_model(f"runs:/{run_id}/mlp_model")

# Save using TF SavedModel format instead of H5
model.export("models/mlp_saved_model")
print("Model saved in SavedModel format")

INFO:tensorflow:Assets written to: models/mlp_saved_model\assets


INFO:tensorflow:Assets written to: models/mlp_saved_model\assets


Saved artifact at 'models/mlp_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 439), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1348267287248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1348267285136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1348267286480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1348267287440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1348267286288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1348267285904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1348267286672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1348267286864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1348267288976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1348267289168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1348267285712: TensorSpec(shape=(), d

In [6]:
import boto3
import os

s3_client = boto3.client("s3", region_name="us-east-2")

for root, dirs, files in os.walk("models/mlp_saved_model"):
    for file in files:
        local_path = os.path.join(root, file)
        s3_key = local_path.replace("\\", "/")
        s3_client.upload_file(local_path, "fraud-detection-mlops-rd", s3_key)
        print(f"Uploaded: {s3_key}")

Uploaded: models/mlp_saved_model/fingerprint.pb
Uploaded: models/mlp_saved_model/saved_model.pb
Uploaded: models/mlp_saved_model/variables/variables.data-00000-of-00001
Uploaded: models/mlp_saved_model/variables/variables.index
